# Acoustic Rain Gauge — Exploratory Data Analysis

**Stage 2** of the project roadmap. Stage 1 (data cleaning) produced 780,725 aligned,
feature-extracted audio clips across 19 monthly folders. This notebook combines them
into a single dataset and explores it before any feature engineering or model training.

Goals:
1. Combine all monthly `cleaned_aligned_data.csv` files and sanity-check the totals
2. Look at class balance and the rainfall distribution
3. Compare acoustic feature distributions between rainy and dry clips
4. Check feature correlations
5. Specifically test the duration anomaly flagged in the README (Jan 2024 / Dec 2023
   are mostly sub-5-second clips, unlike the ~97% of the dataset that's 10-15s)
6. Look at time coverage and outliers
7. Save a single combined dataset for Stage 3 (feature engineering)

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

RAINY_COLOR = "#4C72B0"
DRY_COLOR = "#DD8452"

## 1. Load & Combine All Monthly Datasets

In [ ]:
CLEANED_DATA_DIR = Path(r"D:\arg_cleaned_dataset")

frames = []
for folder in sorted(CLEANED_DATA_DIR.iterdir()):
    if not folder.is_dir() or folder.name.startswith("_"):
        continue
    csv_path = folder / "cleaned_aligned_data.csv"
    if not csv_path.exists():
        continue
    monthly = pd.read_csv(csv_path, parse_dates=["timestamp"])
    monthly["month_folder"] = folder.name
    frames.append(monthly)

data = pd.concat(frames, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
print(f"Loaded {len(frames)} monthly folders, {len(data):,} total rows")
data.head()

In [ ]:
print(f"Total rows      : {len(data):,}")
print(f"Aligned         : {data['is_aligned'].sum():,}")
print(f"Rainy (>0mm)    : {(data['rainfall_mm'] > 0).sum():,}")
print(f"Dry (0mm)       : {(data['rainfall_mm'] == 0).sum():,}")

missing = data.isna().sum()
missing = missing[missing > 0]
print(f"\nColumns with missing values:\n{missing if len(missing) else 'none'}")

# Cross-check against processing_summary.json from the cleaning run
assert len(data) == 780_725, f"Row count mismatch: got {len(data):,}, expected 780,725"
assert (data['rainfall_mm'] > 0).sum() == 114_415, "Rainy count doesn't match processing_summary.json"
print("\nRow counts match processing_summary.json (OK)")

## 2. Class Balance & Rainfall Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

data["rainfall_mm"].gt(0).value_counts().rename({True: "Rainy", False: "Dry"}).plot(
    kind="bar", ax=axes[0], color=[DRY_COLOR, RAINY_COLOR]
)
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Clips")
axes[0].tick_params(axis="x", rotation=0)

rainy_values = data.loc[data["rainfall_mm"] > 0, "rainfall_mm"]
axes[1].hist(rainy_values, bins=50, color=RAINY_COLOR)
axes[1].set_yscale("log")
axes[1].set_title("Rainfall Amount Distribution (rainy clips only, log-scale y)")
axes[1].set_xlabel("rainfall_mm")

plt.tight_layout()
plt.show()

## 3. Acoustic Feature Distributions — Rainy vs Dry

In [ ]:
feature_cols = ["rms", "peak", "par", "spectral_centroid", "spectral_bandwidth",
                "spectral_rolloff", "zero_crossing_rate", "energy_variance"]

plot_df = data.copy()
plot_df["label"] = np.where(plot_df["rainfall_mm"] > 0, "Rainy", "Dry")

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.boxplot(data=plot_df, x="label", y=col, ax=ax, showfliers=False,
                order=["Dry", "Rainy"], palette=[DRY_COLOR, RAINY_COLOR])
    ax.set_title(col)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

## 4. MFCC Distributions — Rainy vs Dry

In [ ]:
mfcc_cols = [f"mfcc_{i}" for i in range(5)]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, mfcc_cols):
    sns.boxplot(data=plot_df, x="label", y=col, ax=ax, showfliers=False,
                order=["Dry", "Rainy"], palette=[DRY_COLOR, RAINY_COLOR])
    ax.set_title(col)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

## 5. Feature Correlation Matrix

In [ ]:
numeric_cols = feature_cols + mfcc_cols + ["duration_sec", "rainfall_mm"]
corr = data[numeric_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Feature Correlation Matrix (incl. rainfall_mm)")
plt.tight_layout()
plt.show()

## 6. Duration Anomaly Check

The README flags that clip duration isn't uniform: ~97% of clips are 10-15s, but
**January 2024 (100%) and December 2023 (96%) are sub-5-second clips**. If acoustic
features are duration-dependent, those two months could look artificially different
to a model for reasons that have nothing to do with rainfall.

In [ ]:
dur_by_month = (data.groupby("month_folder")["duration_category"]
                 .value_counts(normalize=True)
                 .unstack(fill_value=0)
                 .round(2))
dur_by_month

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["rms", "spectral_centroid", "energy_variance"]):
    sns.boxplot(data=data, x="duration_category", y=col, ax=ax, showfliers=False,
                order=["<5s", "5-10s", "10-15s", ">15s"])
    ax.set_title(f"{col} by duration_category")

plt.tight_layout()
plt.show()

print("If these distributions differ mainly by duration_category rather than by",
      "rainfall_mm, duration is a confound that needs correcting in Stage 3",
      "(e.g. per-duration normalization, or excluding/flagging the two short-clip months).")

## 7. Coverage Across Time

In [ ]:
daily = data.set_index("timestamp").resample("D").agg(
    clips=("rainfall_mm", "size"),
    rainy_frac=("rainfall_mm", lambda s: (s > 0).mean()),
)

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
axes[0].plot(daily.index, daily["clips"], color=RAINY_COLOR, lw=0.8)
axes[0].set_ylabel("Clips / day")
axes[0].set_title("Dataset Coverage Over Time")

axes[1].plot(daily.index, daily["rainy_frac"], color=DRY_COLOR, lw=0.8)
axes[1].set_ylabel("Rainy fraction")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

## 8. Outlier Check

In [ ]:
from scipy import stats

print("Extreme outliers (|z-score| > 6) per feature:")
for col in ["rms", "peak", "par", "energy_variance"]:
    z = np.abs(stats.zscore(data[col]))
    n = int((z > 6).sum())
    print(f"  {col:20s}: {n:,} rows ({n / len(data):.3%})")

rms_outliers = data.loc[np.abs(stats.zscore(data["rms"])) > 6,
                         ["audio_filename", "month_folder", "rainfall_mm", "rms", "peak"]]
rms_outliers.head(10)

## 9. Save Combined Dataset for Stage 3

In [ ]:
OUTPUT_PATH = Path("../data/processed/combined_dataset.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved combined dataset: {len(data):,} rows, {data.shape[1]} columns -> {OUTPUT_PATH.resolve()}")

## Summary & Next Steps

- The combined dataset (780,725 rows) matches `processing_summary.json` exactly.
- Class imbalance is real (~14.7% rainy) — confirm from §2 whether class weighting or
  resampling looks necessary once feature separability is assessed in §3-4.
- §6 is the key gate before modeling: if duration confounds features, decide how to
  handle the Jan 2024 / Dec 2023 short-clip months before they enter training.
- `data/processed/combined_dataset.csv` is now the single input for **Stage 3 —
  Feature Engineering** (imbalance handling, scaling, derived features, time-based split).